## Importing libraries

In [ ]:
import numpy as np
import pandas as pd

import json

import glob
import os
from pathlib import Path

import pickle

## Importing data

In [ ]:
# Dictionary containing all the json files imported (json_file_name : json_content)
with open('../raw_data/pickles/4-eumaco.pickle', 'rb') as file:
	json_files_dict = pickle.load(file)

# List of all the json files names
json_filenames = list(json_files_dict)

## Analysis

### Preliminary functions

#### Functions `show_structure` that returns the structure of a field in a json/dictionary

Variant where iterable are NOT analyzed recursively

In [ ]:
def show_structure_simple(data):
    '''Takes in input a dictionary and returns a version of it where:
    - keys are kept unchanged
    - values are substituted with the type name (iterable are NOT analyzed recursively)'''

    result = {}

    for key, val in data.items():
        if isinstance(val, dict):
            result[key] = show_structure_simple(val)  # Recursive call
        else:
            result[key] = type(val).__name__
    return result

Variant where iterable are analyzed recursively

In [ ]:
from collections.abc import Mapping, Sequence

def show_structure_digger(data):
    '''Takes in input a dictionary and returns a version of it where:
    - keys are kept unchanged
    - values are substituted with the type name (iterable are analyzed recursively)'''
    
    if isinstance(data, Mapping):
        result = {}
        for key, val in data.items():
            result[key] = show_structure_digger(val)
        return result

    if isinstance(data, Sequence) and not isinstance(data, (str, bytes)):
        result = []
        for item in data:
            result.append(show_structure_digger(item))
        return result

    return type(data).__name__

#### Function `get_sample` that returns a sample of a field (optional: specify the json file and the instance position)

In [ ]:
def get_sample(json_files_dict, json_filenames, field_to_check, *, as_structure=True, json_filename=None, position=0):
    '''
    Takes in input:
    mandatory:
    - a list of json filenames
    - a dictionary {json_filename: json_file_content}
    - the transmission subfield to check
    optionals:
    - bool as_structure option
    - json filename to read (default=first of json_filenames)
    - transmission field instance's position(default=first one)

    Returns:
    - the structure of the instance, if as_structure=True (default)
    - the actual instance, if as_structure=False
    '''

    if json_filename is None:
        json_filename = json_filenames[0]

    # Gets the required instance
    instance = json_files_dict[json_filename]['transmissions'][field_to_check][0][position]

    if as_structure == False:
        return instance

    # Compute the structure of the required instance
    return show_structure_digger(instance)

#### Function `overview_of_field` that returns overview of the consistency of a field among json files

In [ ]:
def overview_of_field(json_files_dict, json_filenames, field_to_check, sample_instance, *, position=True):
    '''
    Takes in input:
    - a dictionary {json_filename: json_file_content}
    - a list of json filenames
    - the transmission subfield to check
    - a sample instance to check against
    - the position option

    Returns an overview dictionary {'different': {json_file: [position, ...]}, 'missing': [json_files]} that tells:
    - given a json file which have the field different from the sample one:
        - the instance's position within the file, if position=True (default behaviour)
        - the actual instance structure, if position=False
    - which json files misses the transmission subfield to check
    '''

    # The returned object will have this structure:
    # {'different': {json_file: [position, ...]}, 'missing': [json_files]}
    overview = {'different': {}, 'missing': []}

    # Iterates over the jsons
    for json_name in json_filenames:

        # transmissions will be a dict that contains the actual "transmissions" field of the json
        # {field_to_check: content}
        transmissions = json_files_dict[json_name].get('transmissions', {})

        # If the transmission field to check is NOT present:
        # Append the json name to the missing list ...
        # .. and go directly to the next json file 
        if field_to_check not in transmissions:
            overview['missing'].append(json_name)
            continue
        
        # If the transmission field to check is present in the json:

        # Place in instances the complete list of instances of the field_to_check
        instances = transmissions[field_to_check][0]

        # Iterate over the long list of the instances and set:
        # - t_position as the transmission field instance's position
        # - t_instance as the the transmission field's instance itself
        for t_position, t_instance in enumerate(instances):

            # Get the structure of the transmission field's instance
            instance_structure = show_structure_digger(t_instance)

            # If the sample instance is different from the current instance...
            if sample_instance != instance_structure:

                # Set value as:
                # - t_position if show_positions is True (default)
                # - instance_structure if show_positions is False
                value = t_position if position else instance_structure

                # Add the desired value to the different dictionary
                overview['different'].setdefault(json_name, []).append(value)

    return overview

#### Check if the "external" fields except the `transmissions` have the same structure

In [ ]:
def compare_general_jsons(json_files_dict, json_filenames, sample_json_name, *, names=True):
    '''
    Takes in input:
    - a list of json filenames
    - a dictionary {json_filename: json_file_content}
    - a sample instance to check against
    - the json_name option

    Returns an overview list that tells:
    - if names == True (default), the name of the json files that have a structure different from the sample instance
    - if names == False, the actual structure of the different json    '''

    different_general_jsons = []
    sample_json = show_structure_simple(json_files_dict[sample_json_name])
    sample_json.pop('transmissions')

    # Iterates over the jsons
    for json_name in json_filenames:
        current_json = show_structure_simple(json_files_dict[json_name])
        current_json.pop('transmissions')

        if sample_json != current_json:

            if names == True:
                different_general_jsons.append(json_name)
            else:
                different_general_jsons.append(current_json)

    return different_general_jsons

In [ ]:
sample_json = show_structure_simple(json_files_dict['31636'])
sample_json

In [ ]:
kkk = compare_general_jsons(json_files_dict, json_filenames, json_filenames[0], names=True)
kkk

### Running the analysis

Among three transmission fields...

In [ ]:
fields_to_check = json_files_dict[json_filenames[0]]['transmissions'].keys()
list(fields_to_check)

... choose between one of them (`SpO2Transmission`, `WeightTransmission` and `BloodPressureTransmission`)

In [ ]:
field_to_check = 'SpO2Transmission'

Get the sample field

In [ ]:
sample_instance = get_sample(json_files_dict, json_filenames, field_to_check,
    # as_structure=False,
    # json_filename='31893',
    # position=41
)

In [ ]:
sample_instance

#### Run the overview for the specified field

In [ ]:
overview = overview_of_field(json_files_dict, json_filenames, field_to_check, sample_instance,
    # position=False
)

In [ ]:
overview